# Integrate with MCP Catalog on Red Hat OpenShift AI

This notebook registers our MCP servers with the **MCP Gateway** so they appear in the RHOAI **MCP Catalog** (AI Hub → MCP Servers). Once registered, all tools are aggregated behind a single unified endpoint with centralized auth.

**What we'll do:**
1. Install MCP Gateway components (CRDs, controller, RBAC, Gateway listener, MCPGatewayExtension)
2. Create HTTPRoutes for each MCP server (with hostname + port)
3. Create MCPServerRegistration for each MCP server
4. Verify tool discovery and fix known config bugs
5. Test the unified MCP endpoint
6. Register servers in RHOAI Portal via `gen-ai-aa-mcp-servers` ConfigMap
7. Enable MCP Catalog in AI Hub (Dashboard flag + MCP Lifecycle Operator + Catalog Sources)
8. Apply ext_proc workaround for MaaS API (RHOAI ≤ 3.4.0 only)

**Architecture:**

```
┌─────────────────────────────────────────────────────────────────┐
│  MCP Gateway (unified /mcp endpoint)                            │
│  ┌──────────┐   ┌────────┐   ┌───────────┐                     │
│  │  Router   │   │ Broker │   │ Controller│                     │
│  │(ext_proc) │   │        │   │ (watches  │                     │
│  │ parses    │   │aggregates  │ MCPServer │                     │
│  │ JSON-RPC  │   │ tools  │   │  Reg CRs) │                     │
│  └──────────┘   └────────┘   └───────────┘                     │
├─────────────────────────────────────────────────────────────────┤
│  MCPServerRegistration → HTTPRoute → Backend MCP Service        │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────┐ │
│  │ Context7 │ │ SearXNG  │ │Code Sand.│ │Code Srch.│ │ Docs │ │
│  └──────────┘ └──────────┘ └──────────┘ └──────────┘ └──────┘ │
└─────────────────────────────────────────────────────────────────┘
```

**Prerequisites:**
- MCP servers deployed in `mcp-servers` namespace (Phase 1, Step 2)
- MaaS Gateway (`maas-default-gateway`) running in `openshift-ingress`
- `oc` CLI logged in with cluster-admin privileges

**References:**
- [MCP Catalog Blog - Red Hat](https://www.redhat.com/en/blog/mcp-catalog-here-discover-deploy-and-connect-red-hat-openshift-ai)
- [Register On-Prem MCP Servers — RHCL 1.3](https://docs.redhat.com/en/documentation/red_hat_connectivity_link/1.3/html/registering_mcp_servers_and_creating_policies/mcp-gateway-register-on-prem-mcp-servers)
- [MCP Gateway Design Docs](https://github.com/Kuadrant/mcp-gateway/tree/main/docs/design)

In [16]:
import subprocess, json, os
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MCP_NS = "mcp-servers"
GATEWAY_NS = os.getenv("MCP_GATEWAY_NS", "mcp-gateway-system")

print(f"Cluster Domain: {CLUSTER_DOMAIN}")
print(f"MCP Namespace:  {MCP_NS}")
print(f"Gateway NS:     {GATEWAY_NS}")

Cluster Domain: apps.openshift-cluster.sandbox1785.opentlc.com
MCP Namespace:  mcp-servers
Gateway NS:     mcp-gateway-system


## 1. Install MCP Gateway Components

The MCP Gateway is a **Technology Preview** component from Red Hat Connectivity Link (RHCL 1.3+). It consists of three components:
- **Controller** — watches `MCPServerRegistration` CRs and maintains config
- **Broker** — aggregates tools from all registered MCP servers and validates connectivity
- **Router** — routes requests to the correct backend MCP server

> **Note:** On clusters with existing RHCL/Kuadrant/MaaS (e.g., RHOAI 3.4+), the MCP Gateway OLM Subscription may fail due to Authorino CRD schema conflicts. In that case, we install CRDs + controller **manually** from the operator bundle image (see below).

Reference: [Installing the MCP Gateway — RHCL 1.3](https://docs.redhat.com/en/documentation/red_hat_connectivity_link/1.3/html/installing_the_mcp_gateway/mcp-gateway-install)

In [17]:
%%bash
MCP_GW_NS="mcp-gateway-system"
BUNDLE_IMAGE="registry.redhat.io/rhcl-tech-preview/mcp-gateway-operator-bundle@sha256:6e0eb17d728705eddb2849ebcd50f7d95d66a006d32ea571ff7381fb2bf489f3"
CONTROLLER_IMAGE="registry.redhat.io/rhcl-tech-preview/mcp-gateway-rhel9-operator@sha256:d523288fad298f12a8dfc559ff279c06811b32db418c0ffd7a291bfd115fef82"
BROKER_IMAGE="registry.redhat.io/rhcl-tech-preview/mcp-gateway-rhel9@sha256:e2a699c52789a0878b72489d89b2e3c4eb28d411a654c3c89c4b183d75723ac2"

echo "=== 1.1 Check if MCP Gateway is already running ==="
if oc get crd mcpserverregistrations.mcp.kuadrant.io &>/dev/null && \
   oc get deployment mcp-gateway-controller -n ${MCP_GW_NS} &>/dev/null; then
    echo "MCP Gateway CRDs and Controller already present."
    oc get crd | grep mcp.kuadrant.io
    echo ""
    echo "Controller:"
    oc get pods -n ${MCP_GW_NS} -l app=mcp-gateway-controller --no-headers 2>/dev/null || echo "(not found)"
    exit 0
fi

echo "MCP Gateway not installed. Installing manually (avoids Authorino CRD conflict with existing MaaS)..."
echo ""

# Create namespace
oc create ns ${MCP_GW_NS} 2>/dev/null || true

# Step 1: Extract CRDs from the operator bundle image
echo "=== 1.2 Extracting CRDs from bundle image ==="
mkdir -p /tmp/mcp-gw-bundle/manifests
oc image extract ${BUNDLE_IMAGE} --path /manifests/:/tmp/mcp-gw-bundle/manifests/ --confirm 2>/dev/null

# Step 2: Apply CRDs with server-side apply (avoids conflicts)
echo "=== 1.3 Applying MCP Gateway CRDs ==="
oc apply --server-side --force-conflicts -f /tmp/mcp-gw-bundle/manifests/mcp.kuadrant.io_mcpgatewayextensions.yaml
oc apply --server-side --force-conflicts -f /tmp/mcp-gw-bundle/manifests/mcp.kuadrant.io_mcpserverregistrations.yaml
oc apply --server-side --force-conflicts -f /tmp/mcp-gw-bundle/manifests/mcp.kuadrant.io_mcpvirtualservers.yaml
echo ""
oc get crd | grep mcp.kuadrant.io

# Step 3: Create RBAC and deploy controller
echo ""
echo "=== 1.4 Deploying MCP Gateway Controller ==="
oc create serviceaccount mcp-controller -n ${MCP_GW_NS} 2>/dev/null || true

oc apply -f - <<EOF
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: mcp-gateway-controller
rules:
- apiGroups: ["mcp.kuadrant.io"]
  resources: ["*"]
  verbs: ["*"]
- apiGroups: ["gateway.networking.k8s.io"]
  resources: ["gateways", "gateways/status", "httproutes", "httproutes/status", "referencegrants"]
  verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
- apiGroups: [""]
  resources: ["configmaps", "services", "secrets", "serviceaccounts", "namespaces"]
  verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
- apiGroups: ["apps"]
  resources: ["deployments"]
  verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
- apiGroups: [""]
  resources: ["events"]
  verbs: ["create", "patch"]
- apiGroups: ["coordination.k8s.io"]
  resources: ["leases"]
  verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
- apiGroups: ["networking.istio.io"]
  resources: ["envoyfilters"]
  verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
- apiGroups: ["route.openshift.io"]
  resources: ["routes", "routes/custom-host"]
  verbs: ["get", "list", "watch", "create", "update", "patch", "delete"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: mcp-gateway-controller
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: mcp-gateway-controller
subjects:
- kind: ServiceAccount
  name: mcp-controller
  namespace: ${MCP_GW_NS}
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mcp-gateway-controller
  namespace: ${MCP_GW_NS}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: mcp-gateway-controller
  template:
    metadata:
      labels:
        app: mcp-gateway-controller
    spec:
      serviceAccountName: mcp-controller
      containers:
      - name: controller
        image: ${CONTROLLER_IMAGE}
        env:
        - name: RELATED_IMAGE_ROUTER_BROKER
          value: "${BROKER_IMAGE}"
        ports:
        - containerPort: 8080
          name: metrics
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 500m
            memory: 256Mi
EOF

echo ""
echo "Waiting for controller to be ready..."
oc rollout status deployment/mcp-gateway-controller -n ${MCP_GW_NS} --timeout=90s

echo ""
echo "MCP Gateway Controller installed successfully!"
oc get pods -n ${MCP_GW_NS} --no-headers

=== 1.1 Check if MCP Gateway is already running ===
MCP Gateway CRDs and Controller already present.
mcpgatewayextensions.mcp.kuadrant.io                                               2026-06-17T13:39:04Z
mcpserverregistrations.mcp.kuadrant.io                                             2026-06-17T13:39:05Z
mcpvirtualservers.mcp.kuadrant.io                                                  2026-06-17T13:39:06Z

Controller:
mcp-gateway-controller-7575986c65-5ghwm   1/1   Running   9     8d


### 1.4 Add MCP Listener to Gateway

The MCP Gateway needs a dedicated listener on the existing `maas-default-gateway`. We'll add an `mcp` listener that the `MCPGatewayExtension` will target.

In [18]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_NS="mcp-servers"
MCP_GW_NS="mcp-gateway-system"
GATEWAY_NS="openshift-ingress"
GATEWAY_NAME="maas-default-gateway"

echo "=== 1.4a Add MCP listener to ${GATEWAY_NAME} ==="
# Check if 'mcp' listener already exists
EXISTING_LISTENER=$(oc get gateway ${GATEWAY_NAME} -n ${GATEWAY_NS} -o jsonpath='{.spec.listeners[?(@.name=="mcp")].name}' 2>/dev/null)
if [ "${EXISTING_LISTENER}" = "mcp" ]; then
    echo "✓ MCP listener already exists on gateway."
else
    echo "Adding 'mcp' listener to ${GATEWAY_NAME}..."
    oc patch gateway ${GATEWAY_NAME} -n ${GATEWAY_NS} --type='json' -p="[
      {
        \"op\": \"add\",
        \"path\": \"/spec/listeners/-\",
        \"value\": {
          \"name\": \"mcp\",
          \"hostname\": \"mcp.${CLUSTER_DOMAIN}\",
          \"port\": 443,
          \"protocol\": \"HTTPS\",
          \"tls\": {
            \"mode\": \"Terminate\",
            \"certificateRefs\": [{
              \"name\": \"default-ingress-cert\",
              \"kind\": \"Secret\"
            }]
          },
          \"allowedRoutes\": {
            \"namespaces\": {
              \"from\": \"All\"
            }
          }
        }
      }
    ]"
    echo "✓ MCP listener added."
fi

echo ""
echo "=== 1.4b Create ReferenceGrants ==="

# Allow MCPGatewayExtension from mcp-gateway-system to reference Gateway in openshift-ingress
oc apply -f - <<EOF
apiVersion: gateway.networking.k8s.io/v1beta1
kind: ReferenceGrant
metadata:
  name: allow-mcp-gateway-ext
  namespace: ${GATEWAY_NS}
spec:
  from:
    - group: mcp.kuadrant.io
      kind: MCPGatewayExtension
      namespace: ${MCP_GW_NS}
  to:
    - group: gateway.networking.k8s.io
      kind: Gateway
---
apiVersion: gateway.networking.k8s.io/v1beta1
kind: ReferenceGrant
metadata:
  name: allow-mcp-httproutes
  namespace: ${GATEWAY_NS}
spec:
  from:
    - group: gateway.networking.k8s.io
      kind: HTTPRoute
      namespace: ${MCP_GW_NS}
    - group: gateway.networking.k8s.io
      kind: HTTPRoute
      namespace: ${MCP_NS}
  to:
    - group: ""
      kind: Service
    - group: gateway.networking.k8s.io
      kind: Gateway
EOF

echo "✓ ReferenceGrants created."
echo ""
oc get referencegrant -n ${GATEWAY_NS} --no-headers

=== 1.4a Add MCP listener to maas-default-gateway ===
✓ MCP listener already exists on gateway.

=== 1.4b Create ReferenceGrants ===
referencegrant.gateway.networking.k8s.io/allow-mcp-gateway-ext unchanged
referencegrant.gateway.networking.k8s.io/allow-mcp-httproutes unchanged
✓ ReferenceGrants created.

allow-mcp-gateway-ext   8d
allow-mcp-httproutes    8d


### 1.5 Create MCPGatewayExtension

The `MCPGatewayExtension` tells the MCP Gateway controller which Gateway listener to integrate with. Once created, it automatically deploys the `mcp-gateway` pods and creates the `/mcp` HTTPRoute.

In [19]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_GW_NS="mcp-gateway-system"

EXISTING=$(oc get mcpgatewayextension -A --no-headers 2>/dev/null | wc -l)
if [ "$EXISTING" -gt 0 ]; then
    echo "MCPGatewayExtension already exists:"
    oc get mcpgatewayextension -A
    exit 0
fi

echo "Creating MCPGatewayExtension targeting MaaS Gateway (listener: mcp)..."

oc apply -f - <<EOF
apiVersion: mcp.kuadrant.io/v1alpha1
kind: MCPGatewayExtension
metadata:
  name: mcp-gateway-ext
  namespace: ${MCP_GW_NS}
spec:
  targetRef:
    group: gateway.networking.k8s.io
    kind: Gateway
    name: maas-default-gateway
    namespace: openshift-ingress
    sectionName: mcp
  httpRouteManagement: Enabled
EOF

echo ""
echo "Waiting for MCPGatewayExtension to be Ready (up to 60s)..."
oc wait --for=condition=Ready mcpgatewayextension/mcp-gateway-ext -n ${MCP_GW_NS} --timeout=60s

echo ""
echo "Status:"
oc get mcpgatewayextension -n ${MCP_GW_NS}
echo ""
echo "Verify mcp-gateway pods:"
oc get pods -n ${MCP_GW_NS} --no-headers
echo ""
echo "Verify auto-created HTTPRoute:"
oc get httproute mcp-gateway-route -n ${MCP_GW_NS} --no-headers 2>/dev/null || echo "(none yet)"

MCPGatewayExtension already exists:
NAMESPACE            NAME              READY   AGE
mcp-gateway-system   mcp-gateway-ext   True    8d


## 2. Create HTTPRoutes for MCP Servers

HTTPRoutes tell the Gateway how to forward incoming requests to each backend MCP service — without them, the `mcp-gateway` pod has no path to reach the servers.

Each `MCPServerRegistration` requires a `targetRef` pointing to an `HTTPRoute`. The HTTPRoute must:
- Reference the `maas-default-gateway` with `sectionName: mcp`
- Include a `hostname` matching the MCP listener
- Point to the backend MCP service with the correct port

> **Important:** The HTTPRoute MUST have at least one `hostname` — the controller rejects routes without it.

In [20]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_NS="mcp-servers"
MCP_HOSTNAME="mcp.${CLUSTER_DOMAIN}"

# Clean up old HTTPRoutes from previous configurations (if any)
oc delete httproute -n ${MCP_NS} -l legacy-mcp-route=true 2>/dev/null || true
for old_route in $(oc get httproute -n ${MCP_NS} --no-headers 2>/dev/null | grep "mcp-route-" | awk '{print $1}'); do
    oc delete httproute ${old_route} -n ${MCP_NS} 2>/dev/null
    echo "Cleaned up old route: ${old_route}"
done

echo "Creating HTTPRoutes for MCP servers (hostname: ${MCP_HOSTNAME})..."
echo ""

oc apply -f - <<EOF
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: mcp-context7
  namespace: ${MCP_NS}
spec:
  parentRefs:
  - name: maas-default-gateway
    namespace: openshift-ingress
    sectionName: mcp
  hostnames:
  - "${MCP_HOSTNAME}"
  rules:
  - backendRefs:
    - name: mcp-context7
      port: 3001
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: mcp-searxng
  namespace: ${MCP_NS}
spec:
  parentRefs:
  - name: maas-default-gateway
    namespace: openshift-ingress
    sectionName: mcp
  hostnames:
  - "${MCP_HOSTNAME}"
  rules:
  - backendRefs:
    - name: mcp-searxng
      port: 8000
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: mcp-code-sandbox
  namespace: ${MCP_NS}
spec:
  parentRefs:
  - name: maas-default-gateway
    namespace: openshift-ingress
    sectionName: mcp
  hostnames:
  - "${MCP_HOSTNAME}"
  rules:
  - backendRefs:
    - name: mcp-code-sandbox
      port: 3005
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: mcp-codebase-search
  namespace: ${MCP_NS}
spec:
  parentRefs:
  - name: maas-default-gateway
    namespace: openshift-ingress
    sectionName: mcp
  hostnames:
  - "${MCP_HOSTNAME}"
  rules:
  - backendRefs:
    - name: mcp-codebase-search
      port: 8000
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: mcp-repo-docs
  namespace: ${MCP_NS}
spec:
  parentRefs:
  - name: maas-default-gateway
    namespace: openshift-ingress
    sectionName: mcp
  hostnames:
  - "${MCP_HOSTNAME}"
  rules:
  - backendRefs:
    - name: mcp-repo-docs
      port: 8000
EOF

echo ""
echo "HTTPRoutes created:"
oc get httproute -n ${MCP_NS} --no-headers | grep -v "mcp-route-"

No resources found
Creating HTTPRoutes for MCP servers (hostname: mcp.apps.openshift-cluster.sandbox1785.opentlc.com)...

httproute.gateway.networking.k8s.io/mcp-context7 configured
httproute.gateway.networking.k8s.io/mcp-searxng configured
httproute.gateway.networking.k8s.io/mcp-code-sandbox configured
httproute.gateway.networking.k8s.io/mcp-codebase-search configured
httproute.gateway.networking.k8s.io/mcp-repo-docs configured

HTTPRoutes created:
mcp-code-sandbox      ["mcp.apps.openshift-cluster.sandbox1785.opentlc.com"]   8d
mcp-codebase-search   ["mcp.apps.openshift-cluster.sandbox1785.opentlc.com"]   8d
mcp-context7          ["mcp.apps.openshift-cluster.sandbox1785.opentlc.com"]   8d
mcp-repo-docs         ["mcp.apps.openshift-cluster.sandbox1785.opentlc.com"]   8d
mcp-searxng           ["mcp.apps.openshift-cluster.sandbox1785.opentlc.com"]   46m


## 3. Register MCP Servers (MCPServerRegistration)

Create `MCPServerRegistration` CRs that reference our HTTPRoutes. The MCP Gateway controller will:
1. Read the `targetRef` HTTPRoute to discover backend service URL and port
2. Initialize a streamable HTTP connection to the MCP server
3. Call `tools/list` to discover available tools
4. Aggregate all tools into the unified gateway endpoint

**Key fields:**
- `targetRef` — **Required.** References the HTTPRoute pointing to the backend MCP service
- `prefix` — Prepended to tool names to avoid collisions across servers (e.g., `codebase_search_code`)
- `path` — MCP endpoint path on the backend server (default: `/mcp`)
- `category` — Used for tool discovery filtering
- `hint` — Short description shown in the MCP catalog UI

> **Note on tool naming conflicts:** If two servers expose a tool with the same name (e.g., `list_files`), the `mcp-gateway` pod rejects the second server. Use `prefix` to namespace tool names.

In [21]:
%%bash
MCP_NS="mcp-servers"
MCP_GW_NS="mcp-gateway-system"

echo "Creating MCPServerRegistration resources..."
echo ""

oc apply -f - <<EOF
apiVersion: mcp.kuadrant.io/v1alpha1
kind: MCPServerRegistration
metadata:
  name: context7
  namespace: ${MCP_NS}
spec:
  targetRef:
    group: gateway.networking.k8s.io
    kind: HTTPRoute
    name: mcp-context7
  hint: "Up-to-date documentation for popular libraries and frameworks"
  category:
  - "documentation"
  path: "/mcp"
---
apiVersion: mcp.kuadrant.io/v1alpha1
kind: MCPServerRegistration
metadata:
  name: searxng
  namespace: ${MCP_NS}
spec:
  targetRef:
    group: gateway.networking.k8s.io
    kind: HTTPRoute
    name: mcp-searxng
  hint: "Web search capability for real-time information retrieval"
  category:
  - "search"
  path: "/mcp"
---
apiVersion: mcp.kuadrant.io/v1alpha1
kind: MCPServerRegistration
metadata:
  name: code-sandbox
  namespace: ${MCP_NS}
spec:
  targetRef:
    group: gateway.networking.k8s.io
    kind: HTTPRoute
    name: mcp-code-sandbox
  hint: "Safe code execution sandbox for testing and validation"
  category:
  - "development"
  path: "/mcp"
---
apiVersion: mcp.kuadrant.io/v1alpha1
kind: MCPServerRegistration
metadata:
  name: codebase-search
  namespace: ${MCP_NS}
spec:
  targetRef:
    group: gateway.networking.k8s.io
    kind: HTTPRoute
    name: mcp-codebase-search
  prefix: "codebase"
  hint: "Semantic search over internal application source code"
  category:
  - "development"
  - "search"
  path: "/mcp"
---
apiVersion: mcp.kuadrant.io/v1alpha1
kind: MCPServerRegistration
metadata:
  name: repo-docs
  namespace: ${MCP_NS}
spec:
  targetRef:
    group: gateway.networking.k8s.io
    kind: HTTPRoute
    name: mcp-repo-docs
  prefix: "docs"
  hint: "Semantic search over internal documentation and guides"
  category:
  - "documentation"
  - "search"
  path: "/mcp"
EOF

echo ""
echo "MCPServerRegistrations created:"
oc get mcpsr -n ${MCP_NS}

Creating MCPServerRegistration resources...



mcpserverregistration.mcp.kuadrant.io/context7 unchanged
mcpserverregistration.mcp.kuadrant.io/searxng unchanged
mcpserverregistration.mcp.kuadrant.io/code-sandbox unchanged
mcpserverregistration.mcp.kuadrant.io/codebase-search unchanged
mcpserverregistration.mcp.kuadrant.io/repo-docs unchanged

MCPServerRegistrations created:
NAME              PREFIX     TARGET                PATH   READY   TOOLS   CATEGORY                     CREDENTIALS   AGE
code-sandbox                 mcp-code-sandbox      /mcp   True    4       ["development"]                            8d
codebase-search   codebase   mcp-codebase-search   /mcp   True    3       ["development","search"]                   8d
context7                     mcp-context7          /mcp   True    2       ["documentation"]                          8d
repo-docs         docs       mcp-repo-docs         /mcp   True    2       ["documentation","search"]                 8d
searxng                      mcp-searxng           /mcp   True    2   

## 4. Verify Tool Discovery

The controller discovers tools from each backend server. Check `status.conditions` and `status.discoveredTools`.

In [22]:
%%bash
MCP_NS="mcp-servers"
MCP_GW_NS="mcp-gateway-system"

echo "Waiting for tool discovery (up to 60s)..."
for i in $(seq 1 12); do
    ALL_READY=$(oc get mcpsr -n ${MCP_NS} -o jsonpath='{range .items[*]}{.status.conditions[?(@.type=="Ready")].status}{"\n"}{end}' 2>/dev/null | grep -v True | wc -l)
    if [ "$ALL_READY" -eq 0 ]; then
        echo "All servers Ready!"
        break
    fi
    sleep 5
done

echo ""
echo "=== MCPServerRegistration Status ==="
oc get mcpsr -n ${MCP_NS}

echo ""
echo "=== Broker Health ==="
oc logs deployment/mcp-gateway -n ${MCP_GW_NS} --tail=3 2>/dev/null | grep "validation"

echo ""
echo "=== Config Secret (server list) ==="
oc get secret mcp-gateway-config -n ${MCP_GW_NS} -o jsonpath='{.data.config\.yaml}' 2>/dev/null | base64 -d | grep -E "name:|url:" || echo "(not yet generated)"

Waiting for tool discovery (up to 60s)...


All servers Ready!

=== MCPServerRegistration Status ===
NAME              PREFIX     TARGET                PATH   READY   TOOLS   CATEGORY                     CREDENTIALS   AGE
code-sandbox                 mcp-code-sandbox      /mcp   True    4       ["development"]                            8d
codebase-search   codebase   mcp-codebase-search   /mcp   True    3       ["development","search"]                   8d
context7                     mcp-context7          /mcp   True    2       ["documentation"]                          8d
repo-docs         docs       mcp-repo-docs         /mcp   True    2       ["documentation","search"]                 8d
searxng                      mcp-searxng           /mcp   True    2       ["search"]                                 47m

=== Broker Health ===

=== Config Secret (server list) ===
  hostname: mcp.apps.openshift-cluster.sandbox1785.opentlc.com
  name: mcp-servers/context7
  url: http://mcp-context7.mcp-servers.svc.cluster.local:3001/mcp
  h

### 4.1 Troubleshooting: If status shows NotReady

Common issues and fixes:

In [23]:
%%bash
MCP_NS="mcp-servers"
MCP_GW_NS="mcp-gateway-system"

echo "=== Troubleshooting Checklist ==="
echo ""

# Check for NotReady servers
echo "1. NotReady servers:"
oc get mcpsr -n ${MCP_NS} -o json 2>/dev/null | python3 -c "
import sys, json
data = json.load(sys.stdin)
found = False
for item in data.get('items', []):
    conds = item.get('status', {}).get('conditions', [])
    for c in conds:
        if c['type'] == 'Ready' and c['status'] != 'True':
            print(f\"  {item['metadata']['name']}: {c.get('message', 'unknown')}\")
            found = True
if not found:
    print('  All servers Ready!')
" 2>/dev/null

echo ""
echo "2. MCPGatewayExtension status:"
oc get mcpgatewayextension -n ${MCP_GW_NS} --no-headers 2>/dev/null || echo "   Not found"

echo ""
echo "3. Broker logs (errors):"
oc logs deployment/mcp-gateway -n ${MCP_GW_NS} --tail=10 2>/dev/null | grep -i 'error\|fail\|refused' | tail -5 || echo "   No errors in recent logs"

echo ""
echo "4. Controller logs (errors):"
oc logs deployment/mcp-gateway-controller -n ${MCP_GW_NS} --tail=10 2>/dev/null | grep -i 'error\|fail\|invalid' | grep -v "unknown field" | tail -5 || echo "   No errors in recent logs"

echo ""
echo "5. Backend MCP server pods:"
oc get pods -n ${MCP_NS} --no-headers | grep -v "build\|Completed"

=== Troubleshooting Checklist ===

1. NotReady servers:


  All servers Ready!

2. MCPGatewayExtension status:
mcp-gateway-ext   True   8d

3. Broker logs (errors):

4. Controller logs (errors):

5. Backend MCP server pods:
mcp-code-sandbox-78d984cb56-t8fm9      1/1   Running     9                 5d14h
mcp-codebase-search-59778887c7-zxs22   1/1   Running     9                 8d
mcp-context7-5497d6c7c9-nmq4g          1/1   Running     379 (2m56s ago)   5d1h
mcp-repo-docs-d8dfd8d79-898zr          1/1   Running     9                 8d
mcp-searxng-5964cc7b95-724qt           1/1   Running     0                 55m
ocp-mcp-server-997449dc8-ph7h9         1/1   Running     8                 7d21h


### 4.2 Known Bug Fix: Config Secret — Port & Toolsets Format

The MCP Gateway controller generates a config Secret (`mcp-gateway-config`) that the `mcp-gateway` pod mounts at `/config`. If it fails to connect to your servers, check for two known issues:

**Bug 1: `toolsets` format** — When using OLM-based install, the generated config may use TOML table syntax instead of a YAML/string array:

```yaml
# WRONG — causes parse error in mcp-gateway
toolsets:
  core: true

# CORRECT — use a list of strings
toolsets:
- "core"
- "config"
- "openshift"
```

**Bug 2: Missing port** — The config may omit the port number from backend URLs:

```yaml
# WRONG — mcp-gateway can't reach server
url: "https://mcp-code-sandbox.mcp-servers.svc.cluster.local"

# CORRECT — include the service port
url: "https://mcp-code-sandbox.mcp-servers.svc.cluster.local:3005"
```

> **Note:** With our manual installation approach, the controller correctly generates the config including ports from the HTTPRoute `backendRef`. These bugs primarily affect OLM-managed installs. The cell below auto-detects and fixes both issues if present.

In [24]:
%%bash
MCP_GW_NS="mcp-gateway-system"

echo "=== Checking MCP Gateway Config Secret ==="
echo ""

# The controller stores config in a Secret (not ConfigMap)
CONFIG_SECRET="mcp-gateway-config"

if ! oc get secret ${CONFIG_SECRET} -n ${MCP_GW_NS} &>/dev/null; then
    echo "Config secret not found. The controller may not have reconciled yet."
    echo "Wait for MCPServerRegistrations to be processed."
    exit 0
fi

echo "Found: ${MCP_GW_NS}/${CONFIG_SECRET}"
echo ""
echo "--- Config Content ---"
oc get secret ${CONFIG_SECRET} -n ${MCP_GW_NS} -o jsonpath='{.data.config\.yaml}' | base64 -d
echo ""
echo ""

# Bug 1: Check for TOML table toolsets
echo "=== Bug 1 Check (toolsets format) ==="
CONFIG=$(oc get secret ${CONFIG_SECRET} -n ${MCP_GW_NS} -o jsonpath='{.data.config\.yaml}' | base64 -d)
if echo "$CONFIG" | grep -q '\[toolsets\]'; then
    echo "  WARNING: Found [toolsets] table format — needs fix!"
else
    echo "  OK — no TOML table format detected"
fi

echo ""

# Bug 2: Check for missing ports
echo "=== Bug 2 Check (missing port in URLs) ==="
if echo "$CONFIG" | grep 'url:' | grep 'svc.cluster.local' | grep -qv ':[0-9]'; then
    echo "  WARNING: Some URLs are missing port numbers — needs fix!"
    echo "$CONFIG" | grep 'url:' | grep 'svc.cluster.local' | grep -v ':[0-9]'
else
    echo "  OK — all URLs include port numbers"
fi

=== Checking MCP Gateway Config Secret ===

Found: mcp-gateway-system/mcp-gateway-config

--- Config Content ---
servers:
- category:
  - documentation
  hint: Up-to-date documentation for popular libraries and frameworks
  hostname: mcp.apps.openshift-cluster.sandbox1785.opentlc.com
  name: mcp-servers/context7
  state: Enabled
  url: http://mcp-context7.mcp-servers.svc.cluster.local:3001/mcp
- category:
  - development
  hint: Safe code execution sandbox for testing and validation
  hostname: mcp.apps.openshift-cluster.sandbox1785.opentlc.com
  name: mcp-servers/code-sandbox
  state: Enabled
  url: http://mcp-code-sandbox.mcp-servers.svc.cluster.local:3005/mcp
- category:
  - development
  - search
  hint: Semantic search over internal application source code
  hostname: mcp.apps.openshift-cluster.sandbox1785.opentlc.com
  name: mcp-servers/codebase-search
  prefix: codebase
  state: Enabled
  url: http://mcp-codebase-search.mcp-servers.svc.cluster.local:8000/mcp
- category:
  - docu

### 4.3 Auto-Fix: Patch Config Secret (if needed)

If bugs were detected above, run this cell to automatically fix the config and restart `mcp-gateway`:

In [ ]:
%%bash
MCP_GW_NS="mcp-gateway-system"
CONFIG_SECRET="mcp-gateway-config"

if ! oc get secret ${CONFIG_SECRET} -n ${MCP_GW_NS} &>/dev/null; then
    echo "Config secret not found — nothing to fix."
    exit 0
fi

echo "Config Secret: ${MCP_GW_NS}/${CONFIG_SECRET}"

# Backup
oc get secret ${CONFIG_SECRET} -n ${MCP_GW_NS} -o yaml > /tmp/mcp-gw-config-backup.yaml
echo "Backup saved to /tmp/mcp-gw-config-backup.yaml"
echo ""

# Decode current config
CONFIG=$(oc get secret ${CONFIG_SECRET} -n ${MCP_GW_NS} -o jsonpath='{.data.config\.yaml}' | base64 -d)
NEEDS_FIX=false

# Bug 1: Fix [toolsets] table → toolsets: ["core", "config"]
if echo "$CONFIG" | grep -q '\[toolsets\]'; then
    echo "Bug 1 FOUND: [toolsets] TOML table format. Fixing..."
    CONFIG=$(echo "$CONFIG" | python3 -c "
import sys, re
content = sys.stdin.read()
pattern = r'\[toolsets\]\s*\n((?:\w+\s*=\s*\w+\s*\n?)+)'
match = re.search(pattern, content)
if match:
    keys = re.findall(r'(\w+)\s*=\s*true', match.group(1))
    replacement = 'toolsets:\n' + '\n'.join(f'- \"{k}\"' for k in keys) + '\n'
    content = content[:match.start()] + replacement + content[match.end():]
print(content, end='')
")
    NEEDS_FIX=true
    echo "  Fixed."
else
    echo "Bug 1: OK (no TOML table toolsets)"
fi

echo ""

# Bug 2: Fix missing port in URLs
if echo "$CONFIG" | grep 'url:' | grep 'svc.cluster.local' | grep -qv ':[0-9]'; then
    echo "Bug 2 FOUND: URLs missing port numbers. Fixing..."
    CONFIG=$(echo "$CONFIG" | python3 -c "
import sys, re
content = sys.stdin.read()
port_map = {
    'mcp-context7': '3001',
    'mcp-searxng': '8000',
    'mcp-code-sandbox': '3005',
    'mcp-codebase-search': '8000',
    'mcp-repo-docs': '8000',
}
for svc, port in port_map.items():
    pattern = f'({svc}\\.mcp-servers\\.svc\\.cluster\\.local)(/|\")'
    replacement = f'\\1:{port}\\2'
    content = re.sub(pattern, replacement, content)
print(content, end='')
")
    NEEDS_FIX=true
    echo "  Fixed."
else
    echo "Bug 2: OK (all URLs include ports)"
fi

echo ""

# Apply fix if needed
if [ "$NEEDS_FIX" = true ]; then
    echo "Applying fixed config..."
    ENCODED=$(echo "$CONFIG" | base64 -w0 2>/dev/null || echo "$CONFIG" | base64)
    oc patch secret ${CONFIG_SECRET} -n ${MCP_GW_NS} --type=json \
      -p="[{\"op\":\"replace\",\"path\":\"/data/config.yaml\",\"value\":\"${ENCODED}\"}]"

    echo ""
    echo "Restarting mcp-gateway to pick up changes..."
    oc rollout restart deployment/mcp-gateway -n ${MCP_GW_NS}
    echo "mcp-gateway restarted. Wait ~30s then re-verify."
else
    echo "No fixes needed — config looks correct."
fi

## 5. Test the Unified MCP Endpoint

Once all servers are registered and discovered, the gateway provides a single `/mcp` endpoint that aggregates tools from all servers.

In [ ]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_ENDPOINT="https://mcp.${CLUSTER_DOMAIN}/mcp"
TOKEN=$(oc whoami -t)

echo "Unified MCP Gateway Endpoint: ${MCP_ENDPOINT}"
echo ""

echo "=== 5.1 Initialize Session ==="
INIT_RESP=$(curl -sSk -D /tmp/mcp_gw_headers -m 10 -X POST "${MCP_ENDPOINT}" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"lab-test","version":"1.0"}}}')

# Extract session ID from response headers
SESSION_ID=$(grep -i 'mcp-session-id:' /tmp/mcp_gw_headers 2>/dev/null | cut -d' ' -f2 | tr -d '\r')

if [ -n "$SESSION_ID" ]; then
    echo "Session ID: ${SESSION_ID}"
else
    echo "No session ID in headers. Response:"
    echo "${INIT_RESP:0:300}"
fi

echo ""
echo "=== 5.2 Send initialized notification ==="
if [ -n "$SESSION_ID" ]; then
    curl -sSk -m 5 -X POST "${MCP_ENDPOINT}" \
      -H "Authorization: Bearer ${TOKEN}" \
      -H "Content-Type: application/json" \
      -H "mcp-session-id: ${SESSION_ID}" \
      -d '{"jsonrpc":"2.0","method":"notifications/initialized"}' > /dev/null 2>&1
    echo "  Sent notifications/initialized"
fi

echo ""
echo "=== 5.3 List All Aggregated Tools ==="
if [ -n "$SESSION_ID" ]; then
    TOOLS_RESP=$(curl -sSk -m 10 -X POST "${MCP_ENDPOINT}" \
      -H "Authorization: Bearer ${TOKEN}" \
      -H "Content-Type: application/json" \
      -H "Accept: application/json, text/event-stream" \
      -H "mcp-session-id: ${SESSION_ID}" \
      -d '{"jsonrpc":"2.0","id":2,"method":"tools/list"}')

    echo "Aggregated tools from all registered servers:"
    echo "$TOOLS_RESP" | python3 -c "
import sys, json
try:
    data = json.loads(sys.stdin.read())
    tools = data.get('result', {}).get('tools', [])
    print(f'Total tools: {len(tools)}')
    print()
    for t in tools:
        desc = t.get('description', '')[:50]
        print(f'  {t[\"name\"]:<35} {desc}')
except Exception as e:
    print(f'Parse error: {e}')
    print(sys.stdin.read()[:200] if hasattr(sys.stdin, 'read') else '')
" 2>/dev/null || echo "  ${TOOLS_RESP:0:500}"
else
    echo "Cannot list tools without session ID."
    echo "Check that MCP Gateway is correctly configured and mcp-gateway pod is running."
fi

rm -f /tmp/mcp_gw_headers

## 6. Register MCP Servers in RHOAI Portal (GenAI Studio)

The RHOAI 3.4 Dashboard displays MCP servers in the **GenAI Studio** section. The portal reads from a specific ConfigMap (`gen-ai-aa-mcp-servers`) in the `redhat-ods-applications` namespace — it does **not** directly watch `MCPServerRegistration` CRDs.

Each MCP server is defined as a separate key in the ConfigMap (JSON object per key). The BFF (Backend-for-Frontend) uses this to render the MCP Servers tab.

> **Important:** Without this ConfigMap, the GenAI Studio will show "MCP servers ConfigMap may not be deployed yet" error. The `MCPServerRegistration` + MCP Gateway setup (Steps 1-5) provides the unified gateway endpoint, while this ConfigMap makes them visible in the portal UI.

In [25]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_NS="mcp-servers"

echo "=== Creating gen-ai-aa-mcp-servers ConfigMap for RHOAI Portal ==="
echo ""

# Get service ports dynamically
CTX7_PORT=$(oc get svc mcp-context7 -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "3001")
SXG_PORT=$(oc get svc mcp-searxng -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "8000")
SANDBOX_PORT=$(oc get svc mcp-code-sandbox -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "3005")
SEARCH_PORT=$(oc get svc mcp-codebase-search -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "8000")
DOCS_PORT=$(oc get svc mcp-repo-docs -n ${MCP_NS} -o jsonpath='{.spec.ports[0].port}' 2>/dev/null || echo "8000")

oc apply -f - <<EOF
apiVersion: v1
kind: ConfigMap
metadata:
  name: gen-ai-aa-mcp-servers
  namespace: redhat-ods-applications
  labels:
    app: gen-ai
    app.kubernetes.io/part-of: gen-ai-studio
data:
  context7: |
    {
      "name": "context7",
      "displayName": "Context7 - Library Documentation",
      "description": "Provides up-to-date documentation for popular libraries and frameworks",
      "url": "http://mcp-context7.${MCP_NS}.svc.cluster.local:${CTX7_PORT}/mcp",
      "transport": "streamable-http",
      "category": "documentation"
    }
  searxng: |
    {
      "name": "searxng",
      "displayName": "SearXNG - Web Search",
      "description": "Web search capability for real-time information retrieval",
      "url": "http://mcp-searxng.${MCP_NS}.svc.cluster.local:${SXG_PORT}/mcp",
      "transport": "streamable-http",
      "category": "search"
    }
  code-sandbox: |
    {
      "name": "code-sandbox",
      "displayName": "Code Sandbox - Execution Environment",
      "description": "Safe code execution sandbox for testing and validation",
      "url": "http://mcp-code-sandbox.${MCP_NS}.svc.cluster.local:${SANDBOX_PORT}/mcp",
      "transport": "sse",
      "category": "development"
    }
  codebase-search: |
    {
      "name": "codebase-search",
      "displayName": "Codebase Search - Internal Code RAG",
      "description": "Semantic search over internal application source code",
      "url": "http://mcp-codebase-search.${MCP_NS}.svc.cluster.local:${SEARCH_PORT}/mcp",
      "transport": "streamable-http",
      "category": "development"
    }
  repo-docs: |
    {
      "name": "repo-docs",
      "displayName": "Repo Docs - Internal Documentation Q&A",
      "description": "Semantic search over internal documentation and guides",
      "url": "http://mcp-repo-docs.${MCP_NS}.svc.cluster.local:${DOCS_PORT}/mcp",
      "transport": "streamable-http",
      "category": "documentation"
    }
EOF

echo ""
echo "=== Verify ConfigMap ==="
oc get cm gen-ai-aa-mcp-servers -n redhat-ods-applications
echo ""
echo "Portal access:"
echo "  RHOAI Dashboard → GenAI Studio → select namespace → MCP Servers tab"
echo "  URL: https://rhods-dashboard-redhat-ods-applications.${CLUSTER_DOMAIN}"

=== Creating gen-ai-aa-mcp-servers ConfigMap for RHOAI Portal ===

configmap/gen-ai-aa-mcp-servers configured

=== Verify ConfigMap ===
NAME                    DATA   AGE
gen-ai-aa-mcp-servers   5      8d

Portal access:
  RHOAI Dashboard → GenAI Studio → select namespace → MCP Servers tab
  URL: https://rhods-dashboard-redhat-ods-applications.apps.openshift-cluster.sandbox1785.opentlc.com


### 6.1 Verify Portal API

Confirm the GenAI Studio BFF correctly reads the ConfigMap and returns all MCP servers as `healthy`:

In [26]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}
MCP_NS="mcp-servers"

echo "=== MCPServerRegistration Status (CLI) ==="
oc get mcpsr -n ${MCP_NS}

echo ""
echo "=== GenAI Studio Portal API Check ==="
POD=$(oc get pods -n redhat-ods-applications -l app=rhods-dashboard -o jsonpath='{.items[0].metadata.name}')
TOKEN=$(oc whoami -t)

RESP=$(oc exec ${POD} -n redhat-ods-applications -c gen-ai-ui -- \
    /bin/sh -c "curl -sk -H 'x-forwarded-access-token: ${TOKEN}' \
    'https://localhost:8143/gen-ai/api/v1/aaa/mcps?namespace=${MCP_NS}'" 2>/dev/null)

echo "$RESP" | python3 -c "
import sys, json
try:
    data = json.loads(sys.stdin.read())
    d = data.get('data', {})
    print(f'Total servers visible in portal: {d.get(\"total_count\", 0)}')
    print()
    for s in d.get('servers', []):
        print(f'  {s[\"name\"]:<20} {s[\"status\"]:<10} ({s[\"transport\"]})')
    print()
    ci = d.get('config_map_info', {})
    print(f'Source: ConfigMap \"{ci.get(\"name\")}\" in ns \"{ci.get(\"namespace\")}\"')
except Exception as e:
    print(f'Error: {e}')
" 2>/dev/null

echo ""
echo "============================================================"
echo "Unified MCP Gateway: https://mcp.${CLUSTER_DOMAIN}/mcp"
echo "RHOAI Dashboard:     https://rhods-dashboard-redhat-ods-applications.${CLUSTER_DOMAIN}"
echo "  → GenAI Studio → select namespace → MCP Servers tab"

=== MCPServerRegistration Status (CLI) ===
NAME              PREFIX     TARGET                PATH   READY   TOOLS   CATEGORY                     CREDENTIALS   AGE
code-sandbox                 mcp-code-sandbox      /mcp   True    4       ["development"]                            8d
codebase-search   codebase   mcp-codebase-search   /mcp   True    3       ["development","search"]                   8d
context7                     mcp-context7          /mcp   True    2       ["documentation"]                          8d
repo-docs         docs       mcp-repo-docs         /mcp   True    2       ["documentation","search"]                 8d
searxng                      mcp-searxng           /mcp   True    2       ["search"]                                 54m

=== GenAI Studio Portal API Check ===
Total servers visible in portal: 5

  repo-docs            healthy    (streamable-http)
  searxng              healthy    (streamable-http)
  code-sandbox         healthy    (sse)
  codebase-searc

## 7. Enable MCP Catalog in AI Hub

![mcp catalog](https://raw.githubusercontent.com/hyogrin/rhoai-coding-assistant-lab/main/images/mcp_catalog.png)

The RHOAI Dashboard has a dedicated **MCP Catalog** page under **AI Hub** that shows curated MCP servers (Red Hat, Partner, Community) and allows one-click deployment. Three components must be configured:

1. **`mcpCatalog: true`** in `OdhDashboardConfig` — enables the "MCP Catalog" navigation menu
2. **`mcp-catalog-sources` ConfigMap** — tells the model-catalog service where to find MCP catalog YAML files (already pre-populated in RHOAI 3.4, just needs to be activated)
3. **MCP Lifecycle Operator** — installs the `MCPServer` CRD that enables the "Deploy" button and "Deployments" tab

> **Reference:** [The MCP catalog is here — Red Hat Blog](https://www.redhat.com/en/blog/mcp-catalog-here-discover-deploy-and-connect-red-hat-openshift-ai)

In [27]:
%%bash
source ../.env 2>/dev/null || true

echo "=========================================="
echo " 7.1 Enable mcpCatalog flag in Dashboard"
echo "=========================================="

CURRENT=$(oc get odhdashboardconfig odh-dashboard-config -n redhat-ods-applications \
    -o jsonpath='{.spec.dashboardConfig.mcpCatalog}' 2>/dev/null)

if [ "$CURRENT" = "true" ]; then
    echo "✓ mcpCatalog already enabled"
else
    oc patch odhdashboardconfig odh-dashboard-config -n redhat-ods-applications \
        --type=merge -p '{"spec":{"dashboardConfig":{"mcpCatalog":true}}}'
    echo "✓ mcpCatalog flag set to true"
fi

echo ""
echo "=========================================="
echo " 7.2 Populate mcp-catalog-sources"
echo "=========================================="

CURRENT_SOURCES=$(oc get cm mcp-catalog-sources -n rhoai-model-registries \
    -o jsonpath='{.data.sources\.yaml}' 2>/dev/null)

if echo "$CURRENT_SOURCES" | grep -q "redhat_mcp_servers"; then
    echo "✓ mcp-catalog-sources already configured"
else
    oc apply -f - <<'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: mcp-catalog-sources
  namespace: rhoai-model-registries
  labels:
    app: model-catalog
    app.kubernetes.io/component: model-catalog
    app.kubernetes.io/managed-by: model-registry-operator
    app.kubernetes.io/name: model-catalog
    app.kubernetes.io/part-of: model-catalog
    component: model-catalog
data:
  sources.yaml: |
    mcp_catalogs:
      - name: Red Hat MCP
        id: redhat_mcp_servers
        enabled: true
        type: yaml
        properties:
          yamlCatalogPath: /shared-data/redhat-mcp-servers-catalog.yaml
        labels:
          - Red Hat
      - name: Partner MCP
        id: partner_mcp_servers
        enabled: true
        type: yaml
        properties:
          yamlCatalogPath: /shared-data/partner-mcp-servers-catalog.yaml
        labels:
          - Partner
      - name: Community MCP
        id: community_mcp_servers
        enabled: true
        type: yaml
        properties:
          yamlCatalogPath: /shared-data/community-mcp-servers-catalog.yaml
        labels:
          - Community
EOF
    echo "✓ mcp-catalog-sources ConfigMap updated"
    echo "  Restarting model-catalog to pick up changes..."
    oc rollout restart deployment/model-catalog -n rhoai-model-registries
    oc rollout status deployment/model-catalog -n rhoai-model-registries --timeout=120s
fi

echo ""
echo "=========================================="
echo " 7.3 Install MCP Lifecycle Operator"
echo "=========================================="

if oc get crd mcpservers.mcp.x-k8s.io &>/dev/null; then
    echo "✓ MCPServer CRD already exists (MCP Lifecycle Operator installed)"
else
    echo "Installing MCP Lifecycle Operator..."
    oc apply -f https://github.com/kubernetes-sigs/mcp-lifecycle-operator/releases/download/v0.1.0/install.yaml
    echo ""
    echo "Waiting for operator pod..."
    sleep 10
    oc get pods -n mcp-lifecycle-operator-system --no-headers
    echo "✓ MCP Lifecycle Operator installed (MCPServer CRD enables Deploy button)"
fi

echo ""
echo "=========================================="
echo " 7.4 Restart Dashboard (if needed)"
echo "=========================================="

# Dashboard needs restart to pick up mcpCatalog flag change
if [ "$CURRENT" != "true" ]; then
    echo "Restarting dashboard to apply mcpCatalog flag..."
    oc rollout restart deployment/rhods-dashboard -n redhat-ods-applications
    oc rollout status deployment/rhods-dashboard -n redhat-ods-applications --timeout=120s
    echo "✓ Dashboard restarted"
else
    echo "✓ No restart needed (flag was already set)"
fi

echo ""
echo "=========================================="
echo " Result"
echo "=========================================="
echo "✓ AI Hub → MCP Catalog menu should now be visible"
echo "✓ Catalog shows Red Hat / Partner / Community MCP servers"
echo "✓ Deploy button enabled (MCPServer CRD present)"
echo ""
echo "Navigate to: AI Hub → MCP Catalog in the RHOAI Dashboard"

 7.1 Enable mcpCatalog flag in Dashboard
✓ mcpCatalog already enabled

 7.2 Populate mcp-catalog-sources
✓ mcp-catalog-sources already configured

 7.3 Install MCP Lifecycle Operator
✓ MCPServer CRD already exists (MCP Lifecycle Operator installed)

 7.4 Restart Dashboard (if needed)
✓ No restart needed (flag was already set)

 Result
✓ AI Hub → MCP Catalog menu should now be visible
✓ Catalog shows Red Hat / Partner / Community MCP servers
✓ Deploy button enabled (MCPServer CRD present)

Navigate to: AI Hub → MCP Catalog in the RHOAI Dashboard


### 7.5 Deploy from Catalog (Example: OpenShift MCP Server)

When you click **Deploy** in the AI Hub MCP Catalog, a `MCPServer` CR is created. However, there are **known issues** in the developer preview:

> **⚠️ Known Bugs (as of RHOAI 3.4 Developer Preview):**
>
> 1. **`toolsets` format error** — The generated config uses a TOML table format (`[toolsets]` / `core = true`) but the server binary expects a **string array**: `toolsets = ["core", "config", "openshift"]`
> 2. **Missing `port 8080` in config** — Without `--port 8080` argument, the server starts in **stdio mode** and immediately crashes with `read error: EOF`
> 3. The dashboard does **not** auto-create the required `ConfigMap` and `ServiceAccount`

The cell below deploys the **Red Hat OpenShift MCP Server** from the catalog with all workarounds applied:

In [28]:
%%bash
source ../.env 2>/dev/null || true
MCP_NS="mcp-servers"

echo "=========================================="
echo " 7.5 Deploy OpenShift MCP Server from Catalog"
echo "=========================================="

# 1. Create ServiceAccount with cluster view permission
oc create sa mcp-viewer -n ${MCP_NS} 2>/dev/null || true
oc adm policy add-cluster-role-to-user view -z mcp-viewer -n ${MCP_NS} 2>/dev/null

# 2. Create config with correct toolsets format (string array, NOT TOML table)
oc apply -f - <<'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: openshift-mcp-server-config
  namespace: mcp-servers
data:
  config.toml: |
    read_only = true
    disable_destructive = true
    toolsets = ["core", "config", "openshift"]
EOF

# 3. Create MCPServer CR with --port 8080 (without --port, server runs in stdio mode and crashes)
oc apply -f - <<EOF
apiVersion: mcp.x-k8s.io/v1alpha1
kind: MCPServer
metadata:
  name: ocp-mcp-server
  namespace: ${MCP_NS}
  annotations:
    mcp.opendatahub.io/catalog-server: openshift-mcp-server
    mcp.opendatahub.io/display-name: ocp-mcp-server
spec:
  source:
    type: ContainerImage
    containerImage:
      ref: registry.redhat.io/openshift-mcp-beta/openshift-mcp-server-rhel9:0.2
  config:
    port: 8080
    path: /mcp
    arguments:
    - --config
    - /etc/mcp-config/config.toml
    - --port
    - "8080"
    storage:
    - path: /etc/mcp-config
      permissions: ReadOnly
      source:
        type: ConfigMap
        configMap:
          name: openshift-mcp-server-config
  runtime:
    security:
      serviceAccountName: mcp-viewer
EOF

echo ""
echo "Waiting for MCPServer to become Running..."
for i in $(seq 1 12); do
    PHASE=$(oc get mcpserver ocp-mcp-server -n ${MCP_NS} -o jsonpath='{.status.phase}' 2>/dev/null)
    if [ "$PHASE" = "Running" ]; then
        break
    fi
    sleep 5
done

echo ""
oc get mcpserver ocp-mcp-server -n ${MCP_NS}
echo ""
PHASE=$(oc get mcpserver ocp-mcp-server -n ${MCP_NS} -o jsonpath='{.status.phase}' 2>/dev/null)
if [ "$PHASE" = "Running" ]; then
    echo "✓ OpenShift MCP Server deployed successfully!"
    echo "  Endpoint: $(oc get mcpserver ocp-mcp-server -n ${MCP_NS} -o jsonpath='{.status.address.url}')"
else
    echo "⚠ Status: ${PHASE} (may still be starting up)"
    oc get mcpserver ocp-mcp-server -n ${MCP_NS} -o jsonpath='{.status.conditions[0].message}'
    echo ""
fi

 7.5 Deploy OpenShift MCP Server from Catalog
clusterrole.rbac.authorization.k8s.io/view added: "mcp-viewer"
configmap/openshift-mcp-server-config unchanged
mcpserver.mcp.x-k8s.io/ocp-mcp-server unchanged

Waiting for MCPServer to become Running...

NAME             PHASE     IMAGE                                                                  PORT   ADDRESS                                                        AGE
ocp-mcp-server   Running   registry.redhat.io/openshift-mcp-beta/openshift-mcp-server-rhel9:0.2   8080   http://ocp-mcp-server.mcp-servers.svc.cluster.local:8080/mcp   7d22h

✓ OpenShift MCP Server deployed successfully!
  Endpoint: http://ocp-mcp-server.mcp-servers.svc.cluster.local:8080/mcp


## 8. Known Issue Fix: MaaS API ext_proc Blocking (RHOAI ≤ 3.4.0)

> **Known Issue (RHOAIENG-66113):** The MCP Gateway controller creates an EnvoyFilter
> (`mcp-ext-proc-mcp-gateway-system-gateway`) that intercepts **all** port 443 traffic on
> `maas-default-gateway`, not just MCP routes. This causes the `mcp-gateway` ext_proc to
> return `"invalid mcp request"` for non-MCP traffic, breaking the Dashboard API keys page
> and `maas-api` endpoints.
>
> **Fixed in:** RHOAI 3.4.1+ (upstream [opendatahub-io/models-as-a-service#970](https://github.com/opendatahub-io/models-as-a-service/pull/970))
>
> The cell below applies a workaround EnvoyFilter that disables the mcp-gateway ext_proc
> on non-MCP virtual hosts (`maas-api.*` and `maas.*`). Skip this step if running RHOAI ≥ 3.4.1.

In [29]:
%%bash
source ../.env 2>/dev/null || true
CLUSTER_DOMAIN=${CLUSTER_DOMAIN:-$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')}

echo "=== Applying ext_proc workaround for RHOAI ≤ 3.4.0 ==="
echo ""

# Check if the problematic filter exists
if ! oc get envoyfilter mcp-ext-proc-mcp-gateway-system-gateway -n openshift-ingress &>/dev/null; then
    echo "✅ mcp-ext-proc filter not found — likely running RHOAI ≥ 3.4.1. Skipping."
    exit 0
fi

# Check if workaround already applied
if oc get envoyfilter disable-mcp-extproc-non-mcp-vhosts -n openshift-ingress &>/dev/null; then
    echo "✅ Workaround already applied. Skipping."
    exit 0
fi

cat <<EOF | oc apply -f -
apiVersion: networking.istio.io/v1alpha3
kind: EnvoyFilter
metadata:
  name: disable-mcp-extproc-non-mcp-vhosts
  namespace: openshift-ingress
  labels:
    app.kubernetes.io/part-of: maas
    app.kubernetes.io/component: extproc-workaround
  annotations:
    description: "Workaround for RHOAIENG-66113 — disable mcp-gateway ext_proc on non-MCP vhosts. Remove after upgrading to RHOAI 3.4.1+"
spec:
  workloadSelector:
    labels:
      gateway.networking.k8s.io/gateway-name: maas-default-gateway
  configPatches:
  - applyTo: VIRTUAL_HOST
    match:
      context: GATEWAY
      routeConfiguration:
        vhost:
          name: "${CLUSTER_DOMAIN}:443"
    patch:
      operation: MERGE
      value:
        typed_per_filter_config:
          envoy.filters.http.ext_proc:
            "@type": type.googleapis.com/envoy.extensions.filters.http.ext_proc.v3.ExtProcPerRoute
            disabled: true
  - applyTo: VIRTUAL_HOST
    match:
      context: GATEWAY
      routeConfiguration:
        vhost:
          name: "maas-api.${CLUSTER_DOMAIN}:443"
    patch:
      operation: MERGE
      value:
        typed_per_filter_config:
          envoy.filters.http.ext_proc:
            "@type": type.googleapis.com/envoy.extensions.filters.http.ext_proc.v3.ExtProcPerRoute
            disabled: true
  - applyTo: VIRTUAL_HOST
    match:
      context: GATEWAY
      routeConfiguration:
        vhost:
          name: "maas.${CLUSTER_DOMAIN}:443"
    patch:
      operation: MERGE
      value:
        typed_per_filter_config:
          envoy.filters.http.ext_proc:
            "@type": type.googleapis.com/envoy.extensions.filters.http.ext_proc.v3.ExtProcPerRoute
            disabled: true
EOF

echo ""
echo "Waiting for envoy config propagation..."
sleep 8

# Verify the fix
TOKEN=$(oc whoami -t)
echo ""
echo "=== Verification ==="
HTTP_CODE=$(curl -sk -o /dev/null -w "%{http_code}" \
  "https://maas.${CLUSTER_DOMAIN}/maas-api/v1/api-keys/search" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "Content-Type: application/json" \
  -d '{"search":""}')

if [ "$HTTP_CODE" = "200" ]; then
    echo "✅ MaaS API reachable (HTTP ${HTTP_CODE}) — Dashboard API keys page should work."
else
    echo "⚠️  MaaS API returned HTTP ${HTTP_CODE}. Check gateway pod logs."
fi

INF_CODE=$(curl -sk -o /dev/null -w "%{http_code}" \
  "https://maas-api.${CLUSTER_DOMAIN}/v1/models" \
  -H "Authorization: Bearer ${TOKEN}")
echo "   Inference /v1/models: HTTP ${INF_CODE}"

=== Applying ext_proc workaround for RHOAI ≤ 3.4.0 ===

✅ Workaround already applied. Skipping.


## Next: IDE Configuration

IDE configuration for the unified MCP Gateway endpoint requires a **MaaS API key** (generated in Phase 2).

→ See `4_connect_ide_clients.ipynb` for both **Direct Route** (no MaaS) and **MaaS Gateway** (unified) connection modes.

---
## Summary

| Step | Resource | What It Does |
|------|----------|-------------|
| 1 | CRDs + Controller + RBAC | Install MCP Gateway components (manual, avoids Authorino CRD conflict) |
| 1 | Gateway listener + ReferenceGrants | Add `mcp` listener to MaaS Gateway for MCP traffic |
| 1 | `MCPGatewayExtension` | Deploy `mcp-gateway` pod and create unified `/mcp` HTTPRoute |
| 2 | `HTTPRoute` (per server) | Route `mcp-gateway` traffic to each backend MCP service (with correct port) |
| 3 | `MCPServerRegistration` | Register server for tool discovery (targetRef → HTTPRoute, prefix for conflicts) |
| 4 | Verify + Bug Fix | Check tools discovered; auto-fix config Secret if needed |
| 5 | Test | Confirm unified endpoint returns all aggregated tools via JSON-RPC |
| 6 | `gen-ai-aa-mcp-servers` ConfigMap | Register servers in RHOAI Portal (GenAI Studio visibility) |
| 7 | `mcpCatalog` flag + Lifecycle Operator | Enable AI Hub → MCP Catalog menu + Deploy button |
| 8 | EnvoyFilter workaround | Disable mcp-gateway ext_proc on non-MCP vhosts (RHOAI ≤ 3.4.0 only) |

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Tool Aggregation** | All registered servers' tools exposed via single `/mcp` endpoint |
| **Prefix** | Prevents naming collisions — `codebase_search_code`, `docs_search_docs` |
| **Config Secret** | Controller generates `mcp-gateway-config` Secret with all server URLs/ports |
| **Portal ConfigMap** | `gen-ai-aa-mcp-servers` in `redhat-ods-applications` — required for RHOAI Dashboard visibility |
| **Auth Passthrough** | Gateway auth (MaaS API key) applies to all MCP requests uniformly |
| **MCP Catalog** | AI Hub menu — requires `mcpCatalog: true` in OdhDashboardConfig + `mcp-catalog-sources` ConfigMap |
| **MCP Lifecycle Operator** | Installs `MCPServer` CRD — enables "Deploy" button in MCP Catalog UI |

### Troubleshooting Quick Reference

| Problem | Check | Fix |
|---------|-------|----- |
| CRD conflict during install | `oc get installplan -n mcp-gateway-system` | Use manual install (extract CRDs from bundle image) |
| MCPServerRegistration NotReady | `oc logs deploy/mcp-gateway -n mcp-gateway-system` | Check `mcp-gateway` connectivity to backend |
| "must have at least one hostname" | Controller logs | Add `hostnames` field to HTTPRoute |
| Tool name conflict | Broker logs (`conflicting tool names`) | Add `prefix` to MCPServerRegistration |
| Missing port in config | `oc get secret mcp-gateway-config -o yaml` | Verify HTTPRoute `backendRefs` has correct port |
| `toolsets` parse error | Config secret content | Change TOML table → YAML string array |
| MCP not visible in portal | Dashboard logs (`gen-ai-aa-mcp-servers not found`) | Create `gen-ai-aa-mcp-servers` ConfigMap (Step 6) |
| Portal shows parsing error | Dashboard logs (`cannot unmarshal array`) | Use separate keys per server (not JSON array) |
| MCP Catalog menu missing | `oc get odhdashboardconfig` → `mcpCatalog` field | Set `mcpCatalog: true` + restart dashboard |
| Deploy button disabled | `oc get crd mcpservers.mcp.x-k8s.io` | Install MCP Lifecycle Operator |
| Catalog shows empty | `oc get cm mcp-catalog-sources -n rhoai-model-registries` | Populate `mcp_catalogs` entries + restart model-catalog |
| "invalid mcp request" on maas-api | `curl https://maas-api.<domain>/maas-api/health` | Apply Step 8 workaround (RHOAIENG-66113, fixed in 3.4.1) |
| Dashboard "Error loading components" (API keys) | `oc logs deploy/maas-ui -n redhat-ods-applications` | Same ext_proc fix — Step 8 |

## Next Steps

- `4_connect_ide_clients.ipynb` — Test individual server connections from IDEs
- `../3_basic_run/` — Run coding assistant with all tools active
- `../2_maas/` — Enable MaaS for production auth and rate limiting